# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [11]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'llama3.2'
openai = OpenAI()

API key looks good so far


In [12]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [13]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [14]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [8]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [17]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [18]:
def select_relevant_links(url):
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [19]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': ' homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'careers page', 'url': 'https://edwarddonner.com/'}]}

In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [21]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama3.2
Found 4 relevant links


{'links': [{'type': 'Company/About page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'Company/About page', 'url': 'https://edwarddonner.com/'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'}]}

In [23]:
select_relevant_links("https://visionxacademy.vercel.app/")

Selecting relevant links for https://visionxacademy.vercel.app/ by calling llama3.2
Found 6 relevant links


{'links': [{'type': 'About page', 'url': 'https://visionxacademy.vercel.app/'},
  {'type': 'Contact page', 'url': 'https://visionxacademy.vercel.app/'},
  {'type': 'Review/testimonial page',
   'url': 'https://visionxacademy.vercel.app/'},
  {'type': 'Contact page',
   'url': 'https://visionxacademy.vercel.app/#reviews'},
  {'type': 'Company/About page',
   'url': 'https://visionxacademy.vercel.app/#about'},
  {'type': 'Driving lessons contact page',
   'url': 'https://visionxacademy.vercel.app/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [24]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [26]:
print(fetch_page_and_all_relevant_links("https://visionxacademy.vercel.app/"))

Selecting relevant links for https://visionxacademy.vercel.app/ by calling llama3.2
Found 1 relevant links
## Landing Page:

VisionX Academy | Driving Lessons in Macarthur, Sydney

Academy
Why us
Lessons
NSW rules
Reviews
Contact
+61 481 770 888
Macarthur · Camden · Campbelltown
Learn to drive with
confidence
, not just to pass the test.
VisionX Academy is a top-rated local driving school in Sydney's south-west. Friendly 1-on-1 lessons with instructor
Binod Poudel
that turn nervous beginners into safe, road-ready drivers.
Call
+61 481 770 888
Book on WhatsApp
5.0
on Google ·
14
reviews
Open 7 days from 6 AM
50
L
Built-up area · 50 km/h
Why learn with VisionX
A nicer way to learn to drive
Learning to drive should feel exciting, not stressful. Here's why Macarthur learners and their parents choose us.
1-on-1, never rushed
Every lesson is just you and Binod — paced around how you learn, not a checklist.
Safety over shortcuts
We build real roadcraft and lifelong habits, not just enough to 

In [27]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [28]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [29]:
get_brochure_user_prompt("HuggingFace", "https://visionxacademy.vercel.app/")

Selecting relevant links for https://visionxacademy.vercel.app/ by calling llama3.2
Found 6 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nVisionX Academy | Driving Lessons in Macarthur, Sydney\n\nAcademy\nWhy us\nLessons\nNSW rules\nReviews\nContact\n+61 481 770 888\nMacarthur · Camden · Campbelltown\nLearn to drive with\nconfidence\n, not just to pass the test.\nVisionX Academy is a top-rated local driving school in Sydney's south-west. Friendly 1-on-1 lessons with instructor\nBinod Poudel\nthat turn nervous beginners into safe, road-ready drivers.\nCall\n+61 481 770 888\nBook on WhatsApp\n5.0\non Google ·\n14\nreviews\nOpen 7 days from 6 AM\n50\nL\nBuilt-up area · 50 km/h\nWhy learn with VisionX\nA nicer way to learn to drive\nLearning to drive should feel exciting, not stressful. Here's why Macarthur learners and their parents choose us.\n1-on-1, never rushed\nEvery lesson is just

In [36]:
def create_brochure(company_name, url):
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [37]:
create_brochure("visionxacademy", "https://visionxacademy.vercel.app/")

Selecting relevant links for https://visionxacademy.vercel.app/ by calling llama3.2
Found 4 relevant links


**VisionX Academy Brochure**

**Welcome to VisionX Academy**

Learn to drive with confidence, not just to pass the test. VisionX Academy is a top-rated local driving school in Sydney's south-west, known for its friendly and personalized lessons.

**Our Philosophy**

At VisionX Academy, we believe that learning to drive should feel exciting, not stressful. Our instructor, Binod Poudel, provides 1-on-1 lessons that are tailored to your learning style, at a pace that suits you. We focus on building real roadcraft and lifelong habits, rather than just covering the basics.

**What Sets Us Apart**

* **Personalized Lessons**: Every lesson is just you and Binod, so you can learn at your own pace.
* **Safety First**: We prioritize your safety above all else, teaching you how to drive confidently and responsibly.
* **Local Knowledge**: Our lessons take place on local roads and test routes, so you can learn how to drive in the real world.
* **Flexibility**: We offer early and late sessions to accommodate your schedule, as well as lessons on weekends.

** Lessons We Offer**

* **L to P**: We cover the full range of driving lessons, from learner to provisional.
* **Car Control & Observation**: Learn to control your vehicle and observe the road.
* **Confidence Building**: Improve your driving skills and build confidence behind the wheel.
* **Test Preparation**: Get structured preparation for your NSW driving test.

**Join Our Team**

We are always looking for qualified instructors to join our team. If you're passionate about teaching driving and want to make a difference in someone's life, we'd love to hear from you. Check out our careers page to learn more.

**Get in Touch**

Contact us to book a lesson or learn more about our services. We're located in Macarthur, Camden, and Campbelltown, and offer flexible hours to suit your needs.

Phone: +61 481 770 888
Email: [insert email]
Website: [insert website]
Address: 123 Macarthur Drive, Macarthur NSW, 2170

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [38]:
def stream_brochure(company_name, url):
    stream = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [39]:
stream_brochure("visionxacademy", "https://visionxacademy.vercel.app/")

Selecting relevant links for https://visionxacademy.vercel.app/ by calling llama3.2
Found 5 relevant links


**VisionX Academy: Empowering SAFE and CONFIDENT Drivers**

**Our Mission**
At VisionX Academy, we believe that learning to drive should be an empowering experience that fosters confidence, safety, and roadworthiness. We provide personalized 1-on-1 lessons, tailored to meet the unique needs of our students, and help them become capable and independent drivers.

**Our Values**

*   **Friendly and patient teaching**: Our instructor, Binod Poudel, is dedicated to creating a comfortable and supportive learning environment, where every student feels at ease and motivated to learn.
*   **Safety first**: We prioritize safety over shortcuts, focusing on building real roadcraft and lifelong habits that ensure our students become safe and responsible drivers.
*   **Personalized learning**: Every lesson is carefully paced to meet the individual needs and learning style of our students, ensuring they grasp the essential skills and concepts required for driving.

**Our Services**

*   **1-on-1 Driving Lessons**: Personalized sessions tailored to meet the unique needs of our students, from first-time drivers to rusty drivers and those looking to regain their nerve.
*   **Test Preparation**: Structured prep for the NSW driving test, ensuring our students feel confident and prepared to pass.
*   **Confidence Building**: Step-by-step guidance to help our students regain confidence behind the wheel, even in challenging situations.

**Why Learn with VisionX Academy?**

*   **Easy to fit around your schedule**: We offer flexible lesson times, including early morning and late evening sessions, to accommodate busy students.
*   **Logbook boost**: Each 1-hour lesson with our licensed instructor counts as 3 hours in your NSW logbook, helping you accelerate your learning.
*   **Local Macarthur roads**: Our students practice on real-world roads, roundabouts, and test routes, preparing them for the challenges they'll encounter on the roads.

**Join Our Community**
At VisionX Academy, we're passionate about empowering safe and confident drivers. Join us today and discover a better way to learn to drive.

**Contact Us**
+61 481 770 888
[Macarthur · Camden · Campbelltown](mailto:info@visionxacademy.com)
[Book on WhatsApp](https://wa.me/614017707088)

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>